# Privacy Experiments

In [3]:
import numpy as np
from pathlib import Path

from DeMICAF.privacy.dp import compute_feature_histograms, dp_feature_histograms

In [4]:
FEATURES_ROOT = Path("/home/hpestana/Travail/Assets/Features/100k")

DATASETS = ["CheXpert", "MIMIC-CXR", "ChestX-ray8", "PadChest"]
N_BINS = 10

In [5]:
for prior in DATASETS:
    # Get bin edges from the prior dataset
    prior_feats = np.load(FEATURES_ROOT / f"{prior}_using_{prior}.npz", allow_pickle=True)
    prior_feats = prior_feats[prior_feats.files[0]]
    feat_max = np.max(prior_feats[:,1:], axis=0)
    feat_min = np.min(prior_feats[:,1:], axis=0)
    feat_edges = [np.linspace(feat_min[i], feat_max[i], N_BINS + 1) for i in range(feat_min.shape[0])]
    del prior_feats

    clients = [c for c in DATASETS if c != prior]
    histograms = {}
    for client in clients:
        client_feats = np.load(FEATURES_ROOT / f"{client}_using_{prior}.npz", allow_pickle=True)
        client_feats = client_feats[client_feats.files[0]]

        histograms[client] = compute_feature_histograms(client_feats[:,1:], feat_edges)

    private_histograms, sigma = dp_feature_histograms(
        client_histograms=list(histograms.values()),
        epsilon=1.0,
        delta=1e-6,
        seed=42,
    )

    print(len(private_histograms))

512
512
512
512
